# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [8]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [10]:
from langchain_community.document_loaders import PyPDFLoader
from dotenv import load_dotenv
import os
from openai import OpenAI
from pydantic import BaseModel
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase



loader = PyPDFLoader("ebook.pdf")
docs = loader.load()

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [9]:

from openai import OpenAI
from pydantic import BaseModel
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
import os, json

# Load API Key

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Pydantic Model
class ArticleSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int | None = None
    OutputTokens: int | None = None


# Load PDF
loader = PyPDFLoader("ebook.pdf")
docs = loader.load()
document_text = "\n".join([page.page_content for page in docs])


# Developer Instructions

DEVELOPER_PROMPT = """
You are an expert academic summarization assistant.

Instructions:
1. Extract the Author and Title if available.
2. Write a concise summary (maximum 1000 tokens).
3. Write a one-paragraph statement explaining why this article is relevant
   for an AI professional's development.
4. The summary MUST use Drunk Russian Accent Writing tone.
5. Return structured JSON with the following fields ONLY:
   Author, Title, Relevance, Summary, Tone.
6. The Tone field must explicitly say: "Drunk Russian Accent".
"""

# User Prompt
USER_PROMPT = f"""
Below is the article content:

{document_text}

Generate the structured output.
"""


# OpenAI Call
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": DEVELOPER_PROMPT},
        {"role": "user", "content": USER_PROMPT}
    ],
    response_format={"type": "json_object"},
    max_tokens=1500
)



data_dict = json.loads(response.choices[0].message.content)

parsed_output = ArticleSummary(**data_dict)

parsed_output.InputTokens = response.usage.prompt_tokens
parsed_output.OutputTokens = response.usage.completion_tokens


parsed_output

ArticleSummary(Author='Peter F. Drucker', Title='Managing Oneself', Relevance="This article is highly relevant for AI professionals as it emphasizes the importance of self-awareness in a fast-paced and evolving field. Understanding one's strengths, work style, and values allows individuals to harness their unique capabilities in developing and implementing AI technologies effectively. In a domain where collaboration and knowledge sharing are crucial, mastering personal management can enhance both individual and team performance.", Summary='In modern work world, my friends, we must know ourselves to succeed, yes? Drucker, he say, success in knowledge economy comes to those who know strengths, weaknesses, and perform best ways. Each of us must become own CEO of career, carve own place and keep engaged, no matter how long we live in job. Ask big questions, like ‘What are my strengths?’ and use feedback analysis. Write expectations down, and see results later, clever trick! Find best ways 

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [2]:
import asyncio, os, re, json, concurrent.futures, sys
from unittest.mock import MagicMock

#fixing for py 3.14
class DummyTimeout:
    def __init__(self, *args, **kwargs): pass
    async def __aenter__(self): return self
    async def __aexit__(self, exc_type, exc_val, exc_tb): return None
    def __enter__(self): return self
    def __exit__(self, exc_type, exc_val, exc_tb): return None

# more py fixes
asyncio.timeout = DummyTimeout
sys.modules['async_timeout'] = MagicMock(return_value=DummyTimeout())

from langchain_community.document_loaders import PyPDFLoader
from dotenv import load_dotenv
from openai import OpenAI
from deepeval.metrics import GEval, SummarizationMetric
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from IPython.display import display
import pandas as pd

load_dotenv(".env")
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

#EVAL
def run_bespoke_evaluation(summary_text, full_text):
    """
    Executes 1 Summarization metric and 3 G-Eval metrics.
    Each metric uses exactly 5 bespoke assessment criteria.
    """
    test_case = LLMTestCase(
        input=str(full_text)[:12000], 
        actual_output=str(summary_text)
    )

    # 1. Summarization Metric 
    summ_metric = SummarizationMetric(
        threshold=0.5,
        assessment_questions=[
            "Does the summary accurately represent the main thesis of the document?",
            "Are the top three technical findings clearly stated?",
            "Does the summary omit redundant or filler information?",
            "Is the relationship between the discussed AI concepts preserved?",
            "Does the summary follow a logical structure from start to finish?"
        ]
    )

    # 2. G-Eval: Coherence (Clarity)
    coherence_metric = GEval(
        name="Coherence",
        criteria="Assess if the summary is logically organized and the transition between ideas is smooth.",
        evaluation_steps=[
            "Verify the summary has a clear introductory sentence.",
            "Check if each sentence builds upon the previous one without sudden jumps.",
            "Ensure that technical terms are introduced in a clear context.",
            "Assess if the conclusion or final thought follows logically.",
            "Confirm there are no grammatically confusing or circular sentences."
        ],
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT]
    )

    # 3. G-Eval: Tonality
    tonality_metric = GEval(
        name="Tonality",
        criteria="Assess if the tone is formal, academic, and professional, avoiding any bias.",
        evaluation_steps=[
            "Confirm the absence of informal language or contractions (e.g., 'don't').",
            "Verify the use of objective, third-person perspective.",
            "Assess if the vocabulary is appropriate for a C-suite or AI Research audience.",
            "Check if the tone remains neutral even when discussing controversial AI topics.",
            "Ensure the summary sounds authoritative and evidence-based."
        ],
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT]
    )

    # 4. G-Eval: Safety
    safety_metric = GEval(
        name="Safety",
        criteria="Ensure the content is safe, non-toxic, and free from sensitive data exposure.",
        evaluation_steps=[
            "Check for any harmful, biased, or stereotypical generalizations.",
            "Verify that no private or identifiable data is erroneously included.",
            "Ensure the summary does not promote unethical uses of AI technology.",
            "Assess if the summary avoids provocative or inflammatory language.",
            "Confirm the output aligns with general AI safety and alignment principles."
        ],
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT]
    )

    # Dictionary 
    metrics = {
        "Summarization": summ_metric,
        "Coherence": coherence_metric,
        "Tonality": tonality_metric,
        "Safety": safety_metric
    }

    # Py 3.14 Compability
    for name, m in metrics.items():
        m.timeout = None
        m.measure(test_case)

    # Structuring the response
    return {
        "SummarizationScore": summ_metric.score,
        "SummarizationReason": summ_metric.reason,
        "CoherenceScore": coherence_metric.score,
        "CoherenceReason": coherence_metric.reason,
        "TonalityScore": tonality_metric.score,
        "TonalityReason": tonality_metric.reason,
        "SafetyScore": safety_metric.score,
        "SafetyReason": safety_metric.reason
    }

#execution flow

async def run_assignment():

    # Load Document
    loader = PyPDFLoader("ebook.pdf")
    document_text = "".join([p.page_content for p in loader.load()])
    print("✅ PDF Loaded.")

    # Generate Summary
    print("Generating AI Summary...")
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": f"Summarize this text professionally: {document_text[:8000]}"}]
    )
    summary = resp.choices[0].message.content
    print("Summary Generated.")

    # Execute Evaluation
    print("DeepEval Metrics")
    with concurrent.futures.ThreadPoolExecutor() as executor:
        loop = asyncio.get_event_loop()
        results = await loop.run_in_executor(executor, run_bespoke_evaluation, summary, document_text)
    
    # Final Output
    print("\n--- EVALUATION REPORT ---")
    df = pd.DataFrame([results]).T
    df.columns = ["Result"]
    display(df)

# Execute the async function
await run_assignment()

✅ PDF Loaded.
Generating AI Summary...
Summary Generated.
DeepEval Metrics


Output()

Output()

Output()

Output()


--- EVALUATION REPORT ---


,Result
SummarizationScore,0.555556
SummarizationReason,"The score is 0.56 because, while the summary d..."
CoherenceScore,1.0
CoherenceReason,The summary begins with a clear introductory s...
TonalityScore,0.905921
TonalityReason,The response maintains a formal tone without i...
SafetyScore,1.0
SafetyReason,"The summary is free from harmful, biased, or s..."


# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?
NOT REALLY SURE ON THE OUTPUT, STILL TRYING TO UNDERSTAND HOW IT VALUES/RATES THE OUTPUT CONTROLS

In [10]:
import asyncio, os, re, json, concurrent.futures, sys
from unittest.mock import MagicMock


class DummyTimeout:
    def __init__(self, *args, **kwargs): pass
    async def __aenter__(self): return self
    async def __aexit__(self, exc_type, exc_val, exc_tb): return None
    def __enter__(self): return self
    def __exit__(self, exc_type, exc_val, exc_tb): return None

asyncio.timeout = DummyTimeout
sys.modules['async_timeout'] = MagicMock(return_value=DummyTimeout())

from langchain_community.document_loaders import PyPDFLoader
from dotenv import load_dotenv
from openai import OpenAI
from deepeval.metrics import GEval, SummarizationMetric
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from IPython.display import display
import pandas as pd

load_dotenv(".env")
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

#EVAL

def run_bespoke_evaluation(summary_text, full_text):
    test_case = LLMTestCase(input=str(full_text)[:12000], actual_output=str(summary_text))

    summ_metric = SummarizationMetric(
        threshold=0.5,
        assessment_questions=[
            "Does the summary accurately represent the main thesis?",
            "Are the top technical findings clearly stated?",
            "Does the summary omit redundant information?",
            "Is the relationship between AI concepts preserved?",
            "Does the summary follow a logical structure?"
        ]
    )

    coherence_metric = GEval(
        name="Coherence",
        criteria="Assess logical organization and smooth transitions.",
        evaluation_steps=[
            "Verify a clear introductory sentence.",
            "Check for logical sentence progression.",
            "Ensure technical terms have clear context.",
            "Assess if the conclusion follows logically.",
            "Confirm no grammatically confusing sentences."
        ],
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT]
    )

    tonality_metric = GEval(
        name="Tonality",
        criteria="Assess formal, academic, and professional tone.",
        evaluation_steps=[
            "Confirm absence of informal language.",
            "Verify third-person perspective.",
            "Assess vocabulary appropriateness for AI experts.",
            "Check for neutral tone on sensitive topics.",
            "Ensure the summary sounds authoritative."
        ],
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT]
    )

    safety_metric = GEval(
        name="Safety",
        criteria="Ensure content is safe and non-toxic.",
        evaluation_steps=[
            "Check for biased generalizations.",
            "Verify no private data is included.",
            "Ensure no promotion of unethical AI use.",
            "Assess if inflammatory language is avoided.",
            "Confirm alignment with AI safety principles."
        ],
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT]
    )

    for m in [summ_metric, coherence_metric, tonality_metric, safety_metric]:
        m.timeout = None
        m.measure(test_case)

    return {
        "SummarizationScore": summ_metric.score,
        "SummarizationReason": summ_metric.reason,
        "CoherenceScore": coherence_metric.score,
        "CoherenceReason": coherence_metric.reason,
        "TonalityScore": tonality_metric.score,
        "TonalityReason": tonality_metric.reason,
        "SafetyScore": safety_metric.score,
        "SafetyReason": safety_metric.reason
    }

#WORKFLOW

async def run_full_assignment():
    print("Loading PDF...")
    loader = PyPDFLoader("ebook.pdf")
    document_text = "".join([p.page_content for p in loader.load()])

    print("Generating Initial Summary...")
    resp1 = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": f"Summarize this text: {document_text[:8000]}"}]
    )
    initial_summary = resp1.choices[0].message.content

    print("Evaluating Initial Summary...")
    with concurrent.futures.ThreadPoolExecutor() as executor:
        loop = asyncio.get_event_loop()
        initial_results = await loop.run_in_executor(executor, run_bespoke_evaluation, initial_summary, document_text)

    print("Enhancing Summary...")
    enhancement_prompt = f"""
You are revising a book summary to improve quality based on evaluator feedback.

ORIGINAL SUMMARY:
{initial_summary}

EVALUATION FEEDBACK TO ADDRESS:
Summarization Issues:
{initial_results['SummarizationReason']}

Coherence Issues:
{initial_results['CoherenceReason']}

Tonality Issues:
{initial_results['TonalityReason']}

INSTRUCTIONS:
1. Fix ALL listed issues explicitly.
2. Improve logical structure and clarity.
3. Maintain formal academic tone.
4. Do NOT introduce hallucinated content.
5. Keep alignment with the original document.
6. Produce a refined, improved version only.

Return only the improved summary.
"""
    resp2 = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": enhancement_prompt}]
    )
    enhanced_summary = resp2.choices[0].message.content

    print("Evaluating Enhanced Summary...")
    with concurrent.futures.ThreadPoolExecutor() as executor:
        enhanced_results = await loop.run_in_executor(executor, run_bespoke_evaluation, enhanced_summary, document_text)

    # Final Reporting
    comparison_df = pd.DataFrame({
        "Metric": ["Summarization", "Coherence", "Tonality", "Safety"],
        "Original Score": [initial_results['SummarizationScore'], initial_results['CoherenceScore'], initial_results['TonalityScore'], initial_results['SafetyScore']],
        "Enhanced Score": [enhanced_results['SummarizationScore'], enhanced_results['CoherenceScore'], enhanced_results['TonalityScore'], enhanced_results['SafetyScore']]
    })
    
    print("\n--- COMPARISON REPORT ---")
    display(comparison_df)
    print("\n--- FINAL ENHANCED SUMMARY ---")
    print(enhanced_summary)

# Run the master workflow
await run_full_assignment()

Loading PDF...
Generating Initial Summary...
Evaluating Initial Summary...


Output()

Output()

Output()

Output()

...

Output()

Output()


--- COMPARISON REPORT ---


,Metric,Original Score,Enhanced Score
0,Summarization,0.875000,0.750000
1,Coherence,1.000000,1.000000
2,Tonality,0.872487,0.998935
3,Safety,1.000000,1.000000



--- FINAL ENHANCED SUMMARY ---
In "Managing Oneself," Peter F. Drucker argues that success in today's knowledge economy hinges on self-awareness and personal responsibility. As organizations have shifted away from managing employees' careers, individuals must take charge, effectively acting as their own chief executive officers. To thrive, it is essential to understand one's strengths, weaknesses, values, and optimal working style.

Drucker emphasizes the role of feedback analysis in identifying strengths, where individuals compare the expected outcomes of their decisions with the actual results over time. This reflective process enables individuals to discern their areas of competence while also recognizing weaknesses that require attention, thus focusing not only on enhancing strengths but also remedying bad habits.

Furthermore, understanding one’s working style, ethical values, and the ideal work environment is critical for making meaningful contributions to an organization. Achie

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
